# Portfolio consumption report — 2023

Customer-level view of 2023 consumption: meters per region, consumption by segment, solar
penetration, tariff mix, top consumers, monthly trend, and actual vs estimated annual usage.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

meters = pd.read_csv("../data/meters.csv", parse_dates=["signup_date"])
readings = pd.read_csv("../data/meter_readings_daily.csv", parse_dates=["date"])
meters.shape, readings.shape

((300, 7), (107503, 3))

Attach the customer attributes to each daily reading.

In [2]:
data = readings.merge(meters, on="meter_id", how="left")
data.shape

(107503, 9)

## Meters per region

In [3]:
meters_per_region = data.groupby("region").size().sort_values(ascending=False)
meters_per_region

region
London      34325
North       23262
Scotland    20368
Midlands    15759
Wales       11449
london       1077
wales         357
midlands      355
north         351
dtype: int64

Cross-check against the meter master:

In [4]:
meters.groupby("region")["tariff"].count()

region
London      92
Midlands    40
North       64
Scotland    55
Wales       32
london       1
midlands     1
north        1
wales        1
Name: tariff, dtype: int64

## Consumption by segment (total kWh, 2023)

In [5]:
segment = data.groupby(["region", "tariff"])["kwh"].sum().unstack()
segment.round(0)

tariff,Fixed,TOU,Variable
region,,,
London,282334.0,109520.0,139122.0
Midlands,65769.0,73176.0,105223.0
North,167158.0,35681.0,159188.0
Scotland,117101.0,22705.0,93561.0
Wales,144438.0,17449.0,45583.0
london,NaN,NaN,3658.0
midlands,NaN,NaN,3136.0
north,3853.0,NaN,NaN
wales,4762.0,NaN,NaN


In [6]:
print(f"Total across segments: {segment.sum().sum():,.0f} kWh")

Total across segments: 1,593,416 kWh


Average daily kWh per customer type:

In [7]:
data.groupby("customer_type")["kwh"].transform("mean").head()

0    69.66359
1    69.66359
2    69.66359
3    69.66359
4    69.66359
Name: kwh, dtype: float64

So roughly 70 kWh per customer per day.

## Solar penetration by region

In [8]:
data["has_solar"] = data["has_solar"].astype(float)
solar_share = data.groupby("region")["has_solar"].mean().round(3)
solar_share

region
London      0.135
Midlands    0.023
North       0.138
Scotland    0.141
Wales       0.094
london      0.000
midlands    0.000
north       0.000
wales       0.000
Name: has_solar, dtype: float64

## Tariff mix within each region

In [9]:
tariff_mix = pd.crosstab(meters["region"], meters["tariff"], normalize=True).round(3)
tariff_mix

tariff,Fixed,TOU,Variable
region,,,
London,0.164,0.056,0.101
Midlands,0.052,0.031,0.056
North,0.115,0.042,0.066
Scotland,0.105,0.028,0.059
Wales,0.056,0.017,0.038
london,0.000,0.000,0.003
midlands,0.000,0.000,0.003
north,0.003,0.000,0.000
wales,0.003,0.000,0.000


## Top 10 meters by consumption

In [10]:
top10 = readings.nlargest(10, "kwh")
top10.merge(meters[["meter_id", "region", "customer_type"]], on="meter_id")

,meter_id,date,kwh,region,customer_type
0,M100054,2023-12-05,252.687,North,sme
1,M100240,2023-01-21,249.823,Midlands,sme
2,M100236,2023-02-24,247.886,London,sme
3,M100054,2023-12-14,233.521,North,sme
4,M100236,2023-02-18,227.948,London,sme
5,M100015,2023-03-14,227.089,London,sme
6,M100240,2023-11-23,226.085,Midlands,sme
7,M100054,2023-02-20,224.042,North,sme
8,M100107,2023-01-12,219.196,North,sme
9,M100015,2023-02-03,218.988,London,sme


## Monthly trend per tariff (total kWh)

In [11]:
data["month"] = data["date"].dt.month
trend = data.pivot_table(index="month", columns="tariff", values="kwh")
trend.round(1)

tariff,Fixed,TOU,Variable
month,,,
1,20.5,19.7,21.9
2,19.9,18.9,21.2
3,18.1,17.2,19.3
4,15.7,14.4,16.4
5,12.8,12.0,13.6
6,10.5,10.1,11.3
7,10.0,9.3,10.7
8,10.7,10.1,11.5
9,12.9,11.8,13.5


Month-on-month growth:

In [12]:
growth = trend.pct_change(axis=1)
growth.round(3).head(6)

tariff,Fixed,TOU,Variable
month,,,
1,NaN,-0.042,0.114
2,NaN,-0.049,0.125
3,NaN,-0.051,0.123
4,NaN,-0.081,0.141
5,NaN,-0.063,0.139
6,NaN,-0.045,0.129


Overall trend, indexed to January:

In [13]:
monthly = data.groupby("month")["kwh"].sum()
(monthly / monthly.iloc[0] - 1).round(2)

month
1     0.00
2    -0.12
3    -0.12
4    -0.27
5    -0.38
6    -0.50
7    -0.52
8    -0.48
9    -0.40
10   -0.25
11   -0.15
12   -0.03
Name: kwh, dtype: float64

## Actual vs estimated annual consumption

Estimates were re-based in April 2023, so compare against readings from April onwards.

In [14]:
recent = data[data["date"] >= "2023-04-01"]
actual = recent.groupby("meter_id")["kwh"].sum().rename("actual_kwh")
cmp = meters.set_index("meter_id").join(actual)
cmp["ratio"] = cmp["actual_kwh"] / cmp["annual_kwh_estimate"]
cmp["ratio"].describe().round(3)

count    294.000
mean       0.691
std        0.015
min        0.625
25%        0.683
50%        0.692
75%        0.701
max        0.730
Name: ratio, dtype: float64

## Results

In [15]:
print(f"1. Largest region: {meters_per_region.index[0]} with {meters_per_region.iloc[0]:,} meters "
      f"({meters_per_region.iloc[0] / meters_per_region.sum():.0%} of the portfolio)")
print(f"2. Portfolio consumption 2023: {data['kwh'].sum():,.0f} kWh; segment table total {segment.sum().sum():,.0f} kWh")
print(f"3. Solar penetration: highest {solar_share.idxmax()} ({solar_share.max():.1%}), "
      f"lowest {solar_share.idxmin()} ({solar_share.min():.1%})")
print(f"4. TOU share within London: {tariff_mix.loc['London', 'TOU']:.1%}")
print(f"5. Consumption is trending down: {(monthly.loc[7] / monthly.loc[1] - 1):+.0%} from January to July")
print(f"6. Customers use {(cmp['ratio'].mean() - 1):+.0%} vs their annual estimate")

1. Largest region: London with 34,325 meters (32% of the portfolio)
2. Portfolio consumption 2023: 1,695,749 kWh; segment table total 1,593,416 kWh
3. Solar penetration: highest Scotland (14.1%), lowest london (0.0%)
4. TOU share within London: 5.6%
5. Consumption is trending down: -52% from January to July
6. Customers use -31% vs their annual estimate
